# Context-aware toxicity: controlled improvement experiment
Run cells in order after restarting the kernel. This notebook writes a separate candidate checkpoint and reuses the prepared-data embedding cache. Your completed run is saved in notebooks/Context_Aware_Toxicity_Completed_Run.ipynb. See NEXT_STEPS.md for instructions.


In [ ]:
import os
import sys
from pathlib import Path

# Discover the repository without a machine-specific drive or cwd assumption.
start = Path(os.environ.get("TOXICITY_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "configs" / "contextual.json").is_file()
             and (p / "contextual" / "config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Launch Jupyter from the project folder, or set TOXICITY_PROJECT_ROOT to its location.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from contextual.config import load_config, location
cfg = load_config(os.environ.get("TOXICITY_CONFIG", ROOT / "configs" / "contextual_candidate.json"))  # TOXICITY_CONFIG may select another JSON file.
print("Project:", ROOT)
print("Data:", location(cfg, "prepared"))
print("Checkpoint:", location(cfg, "checkpoint"))

# Reload modules after editing Python files in an existing notebook session.
import importlib
import contextual.data
importlib.reload(contextual.data)


## 1. Download authentic parent?child conversations

Cornell's CGA-WIKI supplies reply IDs and **comment-level personal-attack labels**. Section headers are excluded. The corpus-level future-attack annotation is never used as the current target label. The archive and SHA-256 provenance stay in the configured raw directory.

Existing Telugu and Tamil code-mixed parquet files are optional supplemental **single-message** examples. They do not supply missing context labels. Custom threads can be supplied using the documented JSONL schema. No automatic translation or template-generated labels are treated as ground truth.

In [ ]:
from contextual.data import download, prepare
raw_directory = download(cfg)
print("Verified and extracted:", raw_directory)

## Prepare and validate downloaded data
Rerun this cell independently. Invalid ancestry is excluded and counted; no parent links are invented.

In [ ]:
manifest = prepare(cfg)
manifest

## 2. Inspect data and leakage-safe splits

Each sample follows the real reply path, oldest ancestor to target. The most recent configured number of turns is retained. Whole conversations, CGA articles, and any groups sharing message text stay in one split. These are new project splits, not the published benchmark splits. Both classes must occur in each split.

In [ ]:
from collections import Counter
from contextual.data import read_jsonl
examples = read_jsonl(location(cfg, "prepared"))
print("Sources:", Counter(e["source"] for e in examples))
print("Splits:", Counter(e["split"] for e in examples))
print("With ancestry:", sum(len(e["texts"]) > 1 for e in examples))
assert manifest["real_context_examples"] > 0
example = next(e for e in examples if len(e["texts"]) > 1)
{"source": example["source"], "turn_count": len(example["texts"]), "label": example["label"]}

## 3. Train the hierarchical Transformer

A frozen IndicBERT encoder embeds each message independently. Long messages are split into token chunks and pooled by content-token count so the tail is retained. Embedding width comes from the model configuration. A trainable Transformer combines the ancestor vectors with learned positions; learned attention pools the whole reply path. The classifier combines the target alone, the contextual target, and the conversation vector.

The pretrained encoder is frozen for reproducible cached embeddings and modest GPU usage. It is **not** a supervised sentence-similarity model. The conversation Transformer and classifier learn from the supplied labelled data. Early stopping selects validation average precision; the binary decision threshold is fitted on validation F1. No test label is used for training or tuning.

## Load pretrained encoder separately
This is transfer learning: IndicBERT supplies pretrained representations and the conversation classifier learns from labelled threads. The first run downloads model weights. UNEXPECTED keys named cls.* are unused pretraining heads; they are expected when loading the base encoder. Other loading errors must be investigated.

In [ ]:
from contextual.embeddings import TextEncoder, cached_embeddings
encoder = TextEncoder(cfg)
print("Encoder ready:", encoder.device, "dimensions:", encoder.dimension)

## Create and cache embeddings
This is the slowest initial stage. Completed embeddings are reused when the data and encoder settings match. Keep the kernel running until the cache is saved.

In [ ]:
vectors, lookup = cached_embeddings(cfg, examples, encoder)
print("Cached vectors:", tuple(vectors.shape))
del encoder, vectors, lookup
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Train a fair target-only baseline
Character TF-IDF is fit only on training targets. Its decision threshold is selected on validation. The test split matches the neural candidate. This cell can take several minutes.

In [ ]:
from contextual.experiments import train_baseline
baseline_report = train_baseline(cfg)
baseline_report

In [ ]:
from contextual.training import train
report = train(cfg)  # Includes held-out evaluation and saves encoder/tokenizer/head/metadata.
report

## 4. Read measured held-out results

Compare full context with the same trained model given only the target. This ablation is not a separately trained baseline. Report per-source metrics because personal attack, toxicity, and offensiveness differ. A multilingual backbone alone does not establish Telugu or Romanized Telugu accuracy. No calibration claim is made for the sigmoid score.

In [ ]:
import json
report = json.loads(location(cfg, "report").read_text(encoding="utf-8"))
print(json.dumps(report, indent=2))

## 5. Inspect individual and whole-conversation embeddings

This example uses an actual held-out thread. Replace `target` and `parents` with your text. A paragraph can be passed as a single message. Do not invent parent links between unrelated sentences. The model receives no gaming/politics/domain label.

In [ ]:
import numpy as np
from contextual.inference import Predictor
predictor = Predictor(location(cfg, "checkpoint"))
sample = next((e for e in examples if e["split"] == "test" and len(e["texts"]) > 1),
              next(e for e in examples if e["split"] == "test"))
target, parents = sample["texts"][-1], sample["texts"][:-1]
result = predictor.predict(target, history=parents, include_embeddings=True)
print("Individual message vectors:", np.asarray(result["individual_embeddings"]).shape)
print("Contextual message vectors:", np.asarray(result["contextual_embeddings"]).shape)
print("Whole conversation vector:", np.asarray(result["conversation_embedding"]).shape)
{k: v for k, v in result.items() if "embedding" not in k}

## 6. Explicit parent?child inference

For a branching discussion, use `predict_thread`. Only the requested target's actual ancestors are included. Missing parents, duplicate IDs, cycles, and cross-conversation links are rejected. Attention weights describe the model computation; they are not a causal explanation or a measured percentage of intent.

In [ ]:
messages = [dict(id=str(i), parent_id=str(i-1) if i else None,
                 conversation_id=sample["group"], text=text)
            for i, text in enumerate(sample["texts"])]
thread_result = predictor.predict_thread(messages, target_id=messages[-1]["id"])
thread_result

## Files and entry points

- `python run_pipeline.py --all`: download ? prepare ? embed ? train ? evaluate.
- `python run_pipeline.py --predict "Your reply" --parent "Earlier message" --embeddings`.
- `python app/server.py`: local web app; `/docs` exposes the API.
- `python -m pytest tests/test_contextual.py -q`: offline correctness and integration tests.

All paths are repository-relative settings resolved in Python, not relative to the shell's working directory. Local checkpoint inference needs no network. See `README.md` and `docs/CONTEXTUAL_REDESIGN.md` for limitations and custom data format.

Sources: [CGA-WIKI](https://convokit.cornell.edu/documentation/awry.html), [IndicBERT](https://huggingface.co/ai4bharat/IndicBERTv2-MLM-Sam-TLM).

## Compare experiments
Use validation average precision to choose candidates. Test results are descriptive: this test set has already been inspected. Collect a new unseen test set before making final accuracy claims. Context dropout is an experiment and is not guaranteed to improve scores.

In [ ]:
from contextual.experiments import compare_reports
compare_reports(cfg)